### 加载数据

In [8]:
import polars as pl
import pandas as pd
import numpy as np
from skfolio.datasets import load_sp500_dataset
from skfolio.preprocessing import prices_to_returns
from sklearn.model_selection import train_test_split
from ml4t.data.providers.qmt_provider import QmtProvider
from sklearn import set_config
set_config(transform_output="pandas")

In [170]:
xt = QmtProvider(dividend_type='back', timeout=180)

2026-08-16 18:22:23 [debug    ] Rate limiter initialized       max_calls=60 period=60.0 provider=qmt
2026-08-16 18:22:23 [debug    ] HTTP session initialized       max_connections=10 timeout=180
2026-08-16 18:22:23 [debug    ] HTTP session closed           


In [171]:
sh_funds_list = xt.get_stock_list_in_sector("沪市基金")
sz_funds_list = xt.get_stock_list_in_sector("深市基金")
hs_funds_list = xt.get_stock_list_in_sector("沪深基金")

In [172]:
start_date = "2005-01-01"
end_date = "2026-08-16"

In [ ]:
sh_funds_pl = xt.fetch_batch_ohlcv(sh_funds_list,start_date, end_date)
sh_funds_pl.write_parquet(f"sh_funds_{start_date.replace('-','')},{end_date.replace('-','')}.parquet")

2026-08-16 14:12:14 [info     ] Batch fetching from QMT HTTP   end=20260816 period=1d start=20000101 total_symbols=1136
2026-08-16 14:13:09 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=4008 remaining=2442
2026-08-16 14:13:09 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=3921 remaining=2529
2026-08-16 14:13:09 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=6448 remaining=2
2026-08-16 14:13:09 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=4231 remaining=2219
2026-08-16 14:13:09 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=4390 remaining=2060
2026-08-16 14:13:09 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=4146 remaining=2304
2026-08-16 14:13:09 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=4214 remaining=2236
2026-08-16 14:13:09 [warning  ] Dropped filled rows (QMT local data may be inc

In [15]:
sz_funds_pl = xt.fetch_batch_ohlcv(sz_funds_list,start_date, end_date)
sz_funds_pl.write_parquet(f"sz_funds_{start_date.replace('-','')},{end_date.replace('-','')}.parquet")

2026-08-16 14:32:48 [info     ] Batch fetching from QMT HTTP   end=20260816 period=1d start=20000101 total_symbols=1095
2026-08-16 14:34:33 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=6450 remaining=0
2026-08-16 14:34:33 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=6444 remaining=6
2026-08-16 14:34:33 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=6450 remaining=0
2026-08-16 14:34:33 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=6443 remaining=7
2026-08-16 14:34:33 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=6450 remaining=0
2026-08-16 14:34:33 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=6450 remaining=0
2026-08-16 14:34:33 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=6450 remaining=0
2026-08-16 14:34:33 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=6

In [173]:
hs_funds_pl = xt.fetch_batch_ohlcv(hs_funds_list,start_date, end_date)
hs_funds_pl.write_parquet(f"hs_funds_{start_date.replace('-','')},{end_date.replace('-','')}.parquet")

2026-08-16 18:22:54 [info     ] Batch fetching from QMT HTTP   end=20260816 period=1d start=20050101 total_symbols=2231
2026-08-16 18:24:33 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=5250 remaining=0
2026-08-16 18:24:33 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=5244 remaining=6
2026-08-16 18:24:33 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=5250 remaining=0
2026-08-16 18:24:33 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=5243 remaining=7
2026-08-16 18:24:34 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=5250 remaining=0
2026-08-16 18:24:34 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=5250 remaining=0
2026-08-16 18:24:34 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=5250 remaining=0
2026-08-16 18:24:34 [warning  ] Dropped filled rows (QMT local data may be incomplete) dropped=5

In [ ]:
sh_funds_prices = (
    sh_funds_pl.select(["timestamp", "symbol", "close"])
    .pivot(on="symbol", index="timestamp", values="close", aggregate_function="first")
    .sort("timestamp")
)
sh_funds_prices.write_parquet(f"sh_funds_prices.parquet")

sz_funds_prices = (
    sz_funds_pl.select(["timestamp", "symbol", "close"])
    .pivot(on="symbol", index="timestamp", values="close", aggregate_function="first")
    .sort("timestamp")
)
sz_funds_prices.write_parquet(f"sz_funds_prices.parquet")

In [174]:
hs_funds_prices = (
    hs_funds_pl.select(["timestamp", "symbol", "close"])
    .pivot(on="symbol", index="timestamp", values="close", aggregate_function="first")
    .sort("timestamp")
)
hs_funds_prices.write_parquet(f"hs_funds_prices.parquet")

In [166]:
import numpy as np
import pandas as pd

MIN_PRICE = 0.01  # 前复权下低于 1 分钱视为不可用，按你的资产类型调整

def cut_unusable_head(s, min_price=MIN_PRICE):
    ok = s[s > min_price]
    if ok.empty:
        return pd.Series(np.nan, index=s.index, name=s.name)  # 整列都不合格，交给后续筛选剔除
    first_idx = ok.index[0]
    return s.where(s.index >= first_idx)

In [9]:
# 加载数据
prices = pl.read_parquet("hs_funds_prices.parquet").to_pandas()
prices = prices.set_index('timestamp')
prices = prices.ffill()
prices = prices[prices.index.year > 2015]
all_symbols = prices.columns
# 转换为线性收益率
X = prices_to_returns(prices, drop_inceptions_nan=False)

In [3]:
import numpy as np

inf_cols = X.columns[np.isinf(X).any(axis=0)]
print(inf_cols.tolist())

[]


### 预筛选
对初始资产宇宙进行预选择处理，其核心目的是在构建投资组合之前，通过特定的规则筛选出符合条件的资产，从而优化后续的计算效率和模型表现。

In [10]:
from Pre_selection import DropTailCorrelated
from skfolio.pre_selection import SelectKExtremes
from skfolio.pre_selection import DropZeroVariance, DropCorrelated
from skfolio.pre_selection import SelectComplete, SelectNonExpiring, SelectNonDominated

### EqualWeighted基准

In [11]:
from skfolio.optimization import EqualWeighted
from skfolio import Population,MultiPeriodPortfolio
from skfolio import RiskMeasure,PerfMeasure,RatioMeasure

### WalkForward交叉验证
- 数据泄露防护与执行延迟控制
通过 purged_size 参数控制训练集和测试集之间的“清洗”间隔，以模拟真实的交易执行延迟。
```
purged_size=0：训练结束与测试开始无缝衔接。
purged_size >= 1：在训练集末尾和测试集开头之间丢弃指定数量的观测值。例如，purged_size=1 意味着当前时期的决策从下一个时期才开始影响性能。
建议：对于每日定价资产、流动性较差的市场或收盘后结算的数据，建议使用 purged_size >= 1 以更真实地反映执行延迟。
```
- 训练集扩展与尾部数据处理
```
expand_train=True：后续的训练集将包含所有过去的观测值，而不仅仅是固定长度的窗口。
reduce_test=True：即使最后一个测试集的样本数少于 test_size，也会返回该分割。默认情况下，不完整的测试集会被忽略。
```

In [12]:
from skfolio.model_selection import WalkForward

In [ ]:
train_portfolios = MultiPeriodPortfolio()
test_portfolios = MultiPeriodPortfolio()
complete_records = []
nondomin_records = []
tailcorr_records = []
cv = WalkForward(test_size=252, train_size=int(252*3), purged_size=1, reduce_test=True, expand_train=False)
for i, (train_index, test_index) in enumerate(cv.split(X)):
    # 划分训练测试集
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]
    # 完整性筛选
    complete_selector = SelectComplete(drop_assets_with_internal_nan=False)
    X_train = complete_selector.fit_transform(X_train)
    complete_records.append(len(X_train.columns))
    if X_train.empty or X_train.shape[1] < 10:
        continue
    # 0方差筛选
    variance_selector = DropZeroVariance(threshold=1e-8)
    X_train = variance_selector.fit_transform(X_train)
    if X_train.empty or X_train.shape[1] < 10:
        continue
    # 非支配筛选
    nondomin_selector = SelectNonDominated(min_n_assets=10,
                        fitness_measures=[PerfMeasure.MEAN, RiskMeasure.VARIANCE, RatioMeasure.SHARPE_RATIO])
    X_train = nondomin_selector.fit_transform(X_train)
    nondomin_records.append(len(X_train.columns))
    if X_train.empty or X_train.shape[1] < 10:
        continue
    # 相关性筛选
    corrlate_selector = DropCorrelated(threshold=0.1, absolute=False)
    X_train = corrlate_selector.fit_transform(X_train)
    # 尾部相关性筛选
    #tailcorr_selector = DropTailCorrelated(threshold=0.2, quantile=0.10)
    #X_train = tailcorr_selector.fit_transform(X_train)
    tailcorr_records.append(len(X_train.columns))
    if X_train.empty:
        continue
    
    m = EqualWeighted(portfolio_params=dict(name="Fold %d"%i)).fit(X_train)
    train_portfolios.append(m.predict(X_train))
    test_portfolios.append(m.predict(X_test[X_train.columns]))

population_train = Population(train_portfolios)
population_test = Population(test_portfolios)

population_train.set_portfolio_params(tag="Train")
population_test.set_portfolio_params(tag="Test")
population = population_train + population_test

In [15]:
complete_records,nondomin_records,tailcorr_records

([238, 307, 382, 423, 544, 741, 1018, 1165],
 [15, 19, 16, 27, 17, 18, 12, 30],
 [4, 4, 4, 4, 5, 6, 4, 2])

In [16]:
population.plot_measures(
    x=RiskMeasure.ANNUALIZED_STANDARD_DEVIATION,
    y=PerfMeasure.ANNUALIZED_MEAN,
    color_scale=RatioMeasure.ANNUALIZED_SHARPE_RATIO,
    hover_measures=[RiskMeasure.MAX_DRAWDOWN, RatioMeasure.ANNUALIZED_SORTINO_RATIO],
)

In [17]:
population_train.plot_measures(
    x=RiskMeasure.ANNUALIZED_STANDARD_DEVIATION,
    y=PerfMeasure.ANNUALIZED_MEAN,
    color_scale=RatioMeasure.ANNUALIZED_SHARPE_RATIO,
    hover_measures=[RiskMeasure.MAX_DRAWDOWN, RatioMeasure.ANNUALIZED_SORTINO_RATIO],
)

In [18]:
population_test.plot_measures(
    x=RiskMeasure.ANNUALIZED_STANDARD_DEVIATION,
    y=PerfMeasure.ANNUALIZED_MEAN,
    color_scale=RatioMeasure.ANNUALIZED_SHARPE_RATIO,
    hover_measures=[RiskMeasure.MAX_DRAWDOWN, RatioMeasure.ANNUALIZED_SORTINO_RATIO],
)

In [19]:
population_test.plot_cumulative_returns()

In [20]:
test_portfolios.plot_cumulative_returns()

In [21]:
test_portfolios.summary()

Mean                                     0.033%
Annualized Mean                           8.26%
Variance                                0.0031%
Annualized Variance                       0.78%
Semi-Variance                           0.0015%
Annualized Semi-Variance                  0.39%
Standard Deviation                        0.56%
Annualized Standard Deviation             8.82%
Semi-Deviation                            0.39%
Annualized Semi-Deviation                 6.22%
Mean Absolute Deviation                   0.37%
CVaR at 95%                               1.32%
EVaR at 95%                               1.89%
Worst Realization                         3.10%
CDaR at 95%                              13.37%
MAX Drawdown                             16.35%
Average Drawdown                          3.97%
EDaR at 95%                              13.88%
First Lower Partial Moment                0.19%
Ulcer Index                               0.056
Gini Mean Difference                    

In [22]:
X_test.shape,X_train.shape

((57, 2116), (756, 2))

### WalkForward+CombinatorialPurgedCV
组合净化交叉验证生成多个测试路径，以进行更稳健的时间序列模型评估和分布分析。

In [13]:
from skfolio.model_selection import CombinatorialPurgedCV

In [46]:
def nested_walkforward_cpcv(
    X: pd.DataFrame,
    outer_cv: WalkForward | None = None,
    inner_cv: CombinatorialPurgedCV | None = None,
    min_n_assets: int = 10,
    verbose: bool = True,
):
    """WalkForward(外) × CombinatorialPurgedCV(内) 嵌套交叉验证。

    外循环: WalkForward 滚动窗口, 每个 fold 把 train+test 拼成时间连续块;
    内循环: 在该连续块上做 CPCV, 每个组合都走预筛选→拟合→多测试块预测。
    """
    if outer_cv is None:
        outer_cv = WalkForward(
            test_size=252,
            train_size=int(252 * 3),
            purged_size=0,
            reduce_test=True,
            expand_train=False,
        )
    if inner_cv is None:
        inner_cv = CombinatorialPurgedCV(
            n_folds=8, n_test_folds=2, purged_size=0, embargo_size=0
        )
    result = {}
    path_result = {}

    train_portfolios = MultiPeriodPortfolio()   # 内层训练集组合
    test_path_ids = inner_cv.get_path_ids()     # 每个分割中每个测试集的路径 ID
    # 预筛选记录（与参考代码对应，附加 fold/split 标记）
    complete_records, nondomin_records, tailcorr_records = [], [], []
    skip_records = []

    for i, (train_index, test_index) in enumerate(outer_cv.split(X)):
        # ---- 1) train + test 拼成时间连续窗口 ----
        window_index = np.concatenate([train_index, test_index])
        # 若希望窗口严格连续（把外层 purged 掉的那 1 行也纳入）:
        # window_index = np.arange(train_index[0], test_index[-1] + 1)
        X_window = X.iloc[window_index]
        
        inner_result = dict()
        # test path 收集 (时间位置, portfolio)，稍后按时间排序重组
        path_holder = {i:[] for i in range(inner_cv.n_test_paths)}

        for j, (inner_train_idx, inner_test_idx_list) in enumerate(
            inner_cv.split(X_window)
        ):  
            # ---- 2) 相对窗口索引 -> 全局索引 ----
            abs_train_idx = window_index[inner_train_idx]
            X_train = X.iloc[abs_train_idx]

            # ---- 3) 预筛选（只在内层训练块上拟合） ----
            X_train = SelectComplete(
                drop_assets_with_internal_nan=False
            ).fit_transform(X_train)
            complete_records.append((i, j, len(X_train.columns)))
            if X_train.empty or X_train.shape[1] < min_n_assets:
                skip_records.append((i, j, "complete"))
                continue

            X_train = DropZeroVariance(threshold=1e-8).fit_transform(X_train)
            if X_train.empty or X_train.shape[1] < min_n_assets:
                skip_records.append((i, j, "zero_variance"))
                continue

            X_train = SelectNonDominated(
                min_n_assets=min_n_assets,
                fitness_measures=[
                    PerfMeasure.MEAN,
                    RiskMeasure.VARIANCE,
                ],
            ).fit_transform(X_train)
            nondomin_records.append((i, j, len(X_train.columns)))
            if X_train.empty or X_train.shape[1] < min_n_assets:
                skip_records.append((i, j, "nondominated"))
                continue

            X_train = DropCorrelated(threshold=0.1, absolute=False).fit_transform(
                X_train
            )
            # 尾部相关性筛选可按需打开
            # X_train = DropTailCorrelated(threshold=0.2, quantile=0.10).fit_transform(X_train)
            tailcorr_records.append((i, j, len(X_train.columns)))
            if X_train.empty:
                skip_records.append((i, j, "correlated"))
                continue

            # ---- 4) 拟合 ----
            m = EqualWeighted(
                portfolio_params=dict(name=f"WF{i}-CPCV{j}")
            ).fit(X_train)
            train_portfolios.append(m.predict(X_train))


            # ---- 5) 内层测试：每个组合有 n_test_folds 个测试块 ----
            ptfs = []
            for k, inner_test_idx in enumerate(inner_test_idx_list):
                abs_test_idx = window_index[inner_test_idx]
                ptf = m.predict(X.iloc[abs_test_idx][X_train.columns])
                ptf.name = f"WF{i}-CPCV{j}-Test{k}"
                ptfs.append(ptf)
            # ---- 收集测试path
            for path_id,portfolio in zip(test_path_ids[j], ptfs):
                path_holder[path_id].append(portfolio)
            # innner fold result
            fold_result = dict()
            fold_result['train_portfolio'] = m.predict(X_train)
            fold_result['test_population'] = Population(ptfs)
            fold_result['test_mportfolio'] = MultiPeriodPortfolio(ptfs)
            inner_result[j] = fold_result
        result[i] = inner_result
        path_result[i] = path_holder

        if verbose:
            print(
                f"Outer fold {i}: window={len(X_window)} obs | "
                f"inner splits={inner_cv.get_n_splits(X_window)} | "
                f"complete paths={len(path_holder)}"
            )

    return result, path_result

In [19]:
res, paths = nested_walkforward_cpcv(X)

Outer fold 0: window=1008 obs | inner splits=28 | complete paths=28
Outer fold 1: window=1008 obs | inner splits=28 | complete paths=28
Outer fold 2: window=1008 obs | inner splits=28 | complete paths=28
Outer fold 3: window=1008 obs | inner splits=28 | complete paths=28
Outer fold 4: window=1008 obs | inner splits=28 | complete paths=28
Outer fold 5: window=1008 obs | inner splits=28 | complete paths=28
Outer fold 6: window=1008 obs | inner splits=28 | complete paths=28
Outer fold 7: window=814 obs | inner splits=28 | complete paths=28


In [52]:
res[0][0]

{'train_portfolio': <Portfolio WF0-CPCV0>,
 'test_population': <Population([<Portfolio WF0-CPCV0-Test0>, <Portfolio WF0-CPCV0-Test1>])>,
 'test_mportfolio': <MultiPeriodPortfolio 2315873999520>}

In [73]:
Population([res[7][i]['train_portfolio'] for i in range(28)]).plot_cumulative_returns()

Merge all test paths

In [65]:
test_fold_0 = Population([MultiPeriodPortfolio(paths[0][i]) for i in range(7)])
test_fold_0.plot_cumulative_returns()